### Step 1 - Define the IL-2Rβ/γc binding motif

#### Background:
IL-2 has two binding modalities: anti-inflammatory (binds T-regulatory cells (IL-2Rαβγc) → activates them) and pro-inflammatory (binds T-effector and NK cells (IL-2Rβγc) → activates them). The affinity for the latter complex is roughly 100-fold lower, so administered IL-2 is captured preferentially by the suppressive compartment. This blunts anti-tumour efficacy and forces dose escalation, and high-dose IL-2 is severely toxic. To bypass this mechanism, a de novo cytokine, Neo-2/15 (Silva et al., Nature 2019), was engineered to bind only the IL-2Rβγc complex, lacking the α-binding surface entirely rather than having it mutated away. In murine tumour models, Neo-2/15 achieved superior tumour control with reduced toxicity and provoked no anti-drug antibody response, demonstrating de novo design as an alternative to standard mutagenesis.
#### Objective:
Determine, which IL-2 residues are important for the binding to IL-2Rβ/γc, while excluding residues that contribute to IL-2Rα binding. The resulted frozen-geometry will be used for RFdiffusion for the generation of a novel scaffold, reconstructing IL-2Rβ and γc engagement.
#### Methodology:
Run per-residue ΔSASA against each receptor chain, computed pairwise rather than in the assembled quaternary complex. In the full complex, burial cannot be attributed to a specific partner, so, first, compute IL-2 alone, then IL-2+β, IL-2+γc and IL-2+α separately assigns every buried Å² to the chain responsible for it. <br>
Two deliberately asymmetric thresholds are applied. A β/γc contact requires ≥10 Å² of burial, selecting energetically meaningful contributions rather than proximity. An α contact requires only ≥1 Å². The asymmetry is intentional and conservative: a false negative on the α side would import an α-binding residue into the design. <br>
Contacts are then restricted to residues lying on α-helices, assigned from backbone φ/ψ angles. RFdiffusion reproduces the geometry of frozen positions, and a contact on a disordered loop carries geometric information that a scaffold cannot reliably present.
#### Input data:
PDB:2B5I IL-2 with IL-2Rα, IL-2Rβ and γc (Wang, Rickert & Garcia, Science 2005)
UniProt: P60568, P01589, P14784, P31785
#### Not in scope:
No calculation of binding free energy.

In [ ]:
# Environment and directory setup
!pip install -q biopython freesasa

from google.colab import drive
drive.mount('/content/drive')

import os
PROJ = '/content/drive/MyDrive/il2_mimetic'
for d in ['', '/stage1_motif', '/stage2_backbones', '/stage3_seqs',
          '/stage4_fold', '/stage5_complex', '/controls']:
    os.makedirs(PROJ + d, exist_ok=True)
print('project root:', PROJ)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.1/270.1 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 65.6 MB/s eta 0:00:00
Mounted at /content/drive
project root: /content/drive/MyDrive/il2_mimetic


In [ ]:
# Resolve chain identity by local alignment against UniProt rather by the PDB header
import urllib.request
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1
from Bio.Align import PairwiseAligner, substitution_matrices

PDB_ID = '2B5I'
pdb_path = f'{PROJ}/{PDB_ID}.pdb'
if not os.path.exists(pdb_path):
    urllib.request.urlretrieve(
        f'https://files.rcsb.org/download/{PDB_ID}.pdb', pdb_path)
print('structure:', pdb_path)

# reference sequences
UNIPROT = {'IL2': 'P60568', 'IL2Ra': 'P01589',
           'IL2Rb': 'P14784', 'gc': 'P31785'}
ref = {}
for name, acc in UNIPROT.items():
    txt = urllib.request.urlopen(
        f'https://rest.uniprot.org/uniprotkb/{acc}.fasta').read().decode()
    ref[name] = ''.join(txt.split('\n')[1:])
    print(f'{name:6s} {acc}  {len(ref[name])} aa')

# extract the modelled protein sequence of every chain
model = PDBParser(QUIET=True).get_structure(PDB_ID, pdb_path)[0]
chain_seqs = {}
for ch in model:
    res = [r for r in ch if is_aa(r, standard=True)]
    if len(res) < 20:
        continue
    chain_seqs[ch.id] = ''.join(seq1(r.get_resname()) for r in res)

# score each chain against each reference
aligner = PairwiseAligner()
aligner.mode = 'local'
aligner.substitution_matrix = substitution_matrices.load('BLOSUM62')
aligner.open_gap_score, aligner.extend_gap_score = -11, -1

print(f'\n{"chain":>5} {"len":>5}  {"assignment":<8} {"%id":>6}')
CHAIN = {}
for cid, seq in chain_seqs.items():
    scores = {}
    for name, rseq in ref.items():
        aln = aligner.align(seq, rseq)[0]
        a0, a1 = str(aln[0]), str(aln[1])
        scores[name] = 100 * sum(
            x == y and x != '-' for x, y in zip(a0, a1)) / len(seq)
    best = max(scores, key=scores.get)
    CHAIN[best] = cid
    print(f'{cid:>5} {len(seq):>5}  {best:<8} {scores[best]:>5.1f}%')

assert set(CHAIN) == set(UNIPROT), f'missing a chain: {set(CHAIN)}'
print('\nCHAIN =', CHAIN)

structure: /content/drive/MyDrive/il2_mimetic/2B5I.pdb
IL2    P60568  153 aa
IL2Ra  P01589  272 aa
IL2Rb  P14784  551 aa
gc     P31785  369 aa

chain   len  assignment    %id
    A   120  IL2      100.0%
    B   196  IL2Rb     99.0%
    C   191  gc        99.5%
    D   122  IL2Ra    100.0%

CHAIN = {'IL2': 'A', 'IL2Rb': 'B', 'gc': 'C', 'IL2Ra': 'D'}


All four chains are assigned unambiguously (≥99% identity to their UniProt references). The assignment is made by alignment rather than from the PDB header because IL-2Rα and IL-2Rβ differ in length by three residues.

In [ ]:
# Examine deposited SEQRES against observed ATOM residues,
# locate every chain break, and measure how close the flanking
# residues of each α-chain break sit to IL-2

from Bio import SeqIO
from Bio.PDB import NeighborSearch

A = CHAIN['IL2']

# 1. deposited vs observed
print('--- deposited (SEQRES) vs observed (ATOM) ---')
seqres = {r.id.split(':')[-1]: len(r.seq)
          for r in SeqIO.parse(pdb_path, 'pdb-seqres')}
for name, cid in CHAIN.items():
    obs = [r.id[1] for r in model[cid] if is_aa(r, standard=True)]
    print(f'{name:6s} chain {cid}  SEQRES {seqres.get(cid,"?"):>4}  '
          f'observed {len(obs):>4}  numbering {min(obs)}–{max(obs)}')

# 2. where the breaks are
print('\n--- observed residue ranges (gaps = unresolved) ---')
for name, cid in CHAIN.items():
    nums = sorted(r.id[1] for r in model[cid] if is_aa(r, standard=True))
    runs = []
    for n in nums:
        if runs and n - runs[-1][1] == 1: runs[-1][1] = n
        else: runs.append([n, n])
    print(f'{name:6s} ' + ', '.join(f'{s}-{e}' for s, e in runs))

# 3. do any α breaks sit near IL-2?
il2_atoms = [a for r in model[A] if is_aa(r, standard=True) for a in r]
ns = NeighborSearch(il2_atoms)
cid = CHAIN['IL2Ra']
nums = sorted(r.id[1] for r in model[cid] if is_aa(r, standard=True))
gaps = [(nums[i], nums[i+1]) for i in range(len(nums)-1)
        if nums[i+1] - nums[i] > 1]

print('\n--- α chain breaks, and how close their flanks are to IL-2 ---')
if not gaps:
    print('no internal breaks; all unresolved residues are at the termini')
for lo, hi in gaps:
    d = []
    for rn in (lo, hi):
        res = model[cid][rn]
        d.append(min((min((a - b) for b in il2_atoms) for a in res)))
    flag = '  <-- NEAR THE INTERFACE' if min(d) < 10 else ''
    print(f'  missing {lo+1}–{hi-1} ({hi-lo-1} res); '
          f'flanks are {d[0]:.1f} Å and {d[1]:.1f} Å from IL-2{flag}')

--- deposited (SEQRES) vs observed (ATOM) ---
IL2    chain A  SEQRES  133  observed  120  numbering 6–133
IL2Rb  chain B  SEQRES  214  observed  196  numbering 6–207
gc     chain C  SEQRES  199  observed  191  numbering 34–224
IL2Ra  chain D  SEQRES  217  observed  122  numbering 1–165

--- observed residue ranges (gaps = unresolved) ---
IL2    6-75, 80-98, 103-133
IL2Rb  6-24, 31-207
gc     34-224
IL2Ra  1-47, 53-64, 103-165

--- α chain breaks, and how close their flanks are to IL-2 ---
  missing 48–52 (5 res); flanks are 10.3 Å and 16.3 Å from IL-2
  missing 65–102 (38 res); flanks are 5.9 Å and 17.3 Å from IL-2  <-- NEAR THE INTERFACE


In [ ]:
# Run per-residue ΔSASA against each receptor computed pairwise,
# so burial is attributable to a single partner,
# define asymmetric thresholds (≥10 Å² for β/γc, ≥1 Å² for α),
# because missing β/γc contact weakens the designed interface,
# missing an α contact silently imports α-binding capability,
# defeating the purpose of the design

import freesasa, pandas as pd
from Bio.PDB import PDBIO, Select

# --- parameters (these go in params.json) ---
CONTACT_CUTOFF = 10.0   # Å² buried → counts as a β/γc contact
ALPHA_CUTOFF   = 1.0    # Å² buried → counts as an α contact (deliberately strict)
GAP_TOLERANCE  = 3      # merge contacts separated by ≤ this many residues

A = CHAIN['IL2']

class Sel(Select):
    def __init__(self, chains): self.chains = set(chains)
    def accept_chain(self, c):  return c.id in self.chains
    def accept_residue(self, r): return is_aa(r, standard=True)
    def accept_atom(self, a):    return a.element != 'H'

io = PDBIO(); io.set_structure(model)

def il2_sasa(chains, tag):
    """Per-residue SASA of the IL-2 chain in the given subset."""
    p = f'{PROJ}/stage1_motif/_tmp_{tag}.pdb'
    io.save(p, Sel(chains))
    areas = freesasa.calc(freesasa.Structure(p)).residueAreas()
    return {k: v.total for k, v in areas[A].items()}

free   = il2_sasa([A],                  'il2_alone')
with_b = il2_sasa([A, CHAIN['IL2Rb']],  'il2_beta')
with_c = il2_sasa([A, CHAIN['gc']],     'il2_gc')
with_a = il2_sasa([A, CHAIN['IL2Ra']],  'il2_alpha')

resname = {str(r.id[1]): r.get_resname()
           for r in model[A] if is_aa(r, standard=True)}

rows = []
for k in free:
    rows.append({
        'resnum':  int(k),
        'resname': resname.get(k, '???'),
        'sasa_free': round(free[k], 1),
        'd_beta':  round(free[k] - with_b.get(k, free[k]), 1),
        'd_gc':    round(free[k] - with_c.get(k, free[k]), 1),
        'd_alpha': round(free[k] - with_a.get(k, free[k]), 1),
    })

df = pd.DataFrame(rows).sort_values('resnum').reset_index(drop=True)
df['beta_contact']  = df.d_beta  >= CONTACT_CUTOFF
df['gc_contact']    = df.d_gc    >= CONTACT_CUTOFF
df['alpha_contact'] = df.d_alpha >= ALPHA_CUTOFF
df['motif'] = (df.beta_contact | df.gc_contact) & ~df.alpha_contact

df.to_csv(f'{PROJ}/stage1_motif/il2_interface_dsasa.csv', index=False)

# --- summary ---
n_b, n_c = df.beta_contact.sum(), df.gc_contact.sum()
n_a      = df.alpha_contact.sum()
overlap  = ((df.beta_contact | df.gc_contact) & df.alpha_contact).sum()
print(f'IL-2 residues modelled : {len(df)}')
print(f'IL-2Rβ contacts        : {n_b}')
print(f'γc contacts            : {n_c}')
print(f'IL-2Rα contacts        : {n_a}')
print(f'β/γc contacts also touching α (EXCLUDED) : {overlap}')
print(f'MOTIF residues         : {df.motif.sum()}\n')

# --- contiguous segments (what becomes the RFdiffusion contig) ---
sel, segs = sorted(df.loc[df.motif, 'resnum']), []
for r in sel:
    if segs and r - segs[-1][1] <= GAP_TOLERANCE:
        segs[-1][1] = r
    else:
        segs.append([r, r])

print('Frozen segments:')
total = 0
for s, e in segs:
    core = df[(df.resnum >= s) & (df.resnum <= e) & df.motif]
    print(f'  A{s}-{e}  ({e-s+1} res, {len(core)} contacts, '
          f'ΔSASA sum {core.d_beta.sum()+core.d_gc.sum():.0f} Å²)')
    total += e - s + 1
print(f'\nTotal frozen: {total} residues in {len(segs)} segments')
print('Contig draft:', '/'.join(f'A{s}-{e}' for s, e in segs))

IL-2 residues modelled : 120
IL-2Rβ contacts        : 14
γc contacts            : 12
IL-2Rα contacts        : 19
β/γc contacts also touching α (EXCLUDED) : 0
MOTIF residues         : 24

Frozen segments:
  A12-23  (12 res, 9 contacts, ΔSASA sum 428 Å²)
  A81-95  (15 res, 7 contacts, ΔSASA sum 355 Å²)
  A110-110  (1 res, 1 contacts, ΔSASA sum 27 Å²)
  A119-119  (1 res, 1 contacts, ΔSASA sum 28 Å²)
  A123-133  (11 res, 6 contacts, ΔSASA sum 296 Å²)

Total frozen: 40 residues in 5 segments
Contig draft: A12-23/A81-95/A110-110/A119-119/A123-133


In [ ]:
# Quantify the α footprint and test,
# whether the unresolved α loop (residues 65–102) could plausibly have contacted
# any motif residue, since the zero-overlap result is otherwise conditional
# on an incomplete structure

# --- α footprint sanity check + shadow of the missing loop ---
alpha_res = df[df.alpha_contact]
print(f'α footprint: {len(alpha_res)} IL-2 residues, '
      f'{alpha_res.d_alpha.sum():.0f} Å² buried')
print('positions:', sorted(alpha_res.resnum.tolist()))

# IL-2 residues sitting in the shadow of the unresolved 65–102 stretch
flank = model[CHAIN['IL2Ra']][64]
shadow = []
for r in model[A]:
    if not is_aa(r, standard=True): continue
    d = min(min(a - b for b in flank) for a in r)
    if d < 12.0: shadow.append(r.id[1])

risky = sorted(set(shadow) & set(df.loc[df.motif, 'resnum']))
print(f'\nMotif residues within 12 Å of the gap flank: {risky}')
print('→ these are the only ones the unresolved α loop could plausibly have covered')

α footprint: 19 IL-2 residues, 849 Å² buried
positions: [34, 35, 38, 41, 42, 43, 44, 45, 61, 62, 64, 65, 68, 69, 71, 72, 74, 107, 109]

Motif residues within 12 Å of the gap flank: [110]
→ these are the only ones the unresolved α loop could plausibly have covered


In [ ]:
# Assign helices from backbone φ/ψ and retain only contacts lying on them

from Bio.PDB.Polypeptide import PPBuilder
import math

MIN_HELIX = 4   # consecutive residues needed to call a helix
PAD       = 0   # residues to extend each frozen segment beyond its contacts

# --- backbone phi/psi → helix assignment (dependency-free) ---
ss = {}
for pp in PPBuilder().build_peptides(model[A]):
    for res, (phi, psi) in zip(pp, pp.get_phi_psi_list()):
        if phi is None or psi is None:
            ss[res.id[1]] = '-'
        else:
            p, s = math.degrees(phi), math.degrees(psi)
            ss[res.id[1]] = 'H' if (-100 < p < -30 and -80 < s < -5) else '-'

# keep only runs of >= MIN_HELIX
nums = sorted(ss)
runs, cur = [], []
for n in nums:
    if ss[n] == 'H' and (not cur or n - cur[-1] == 1):
        cur.append(n)
    else:
        if len(cur) >= MIN_HELIX: runs.append((cur[0], cur[-1]))
        cur = [n] if ss[n] == 'H' else []
if len(cur) >= MIN_HELIX: runs.append((cur[0], cur[-1]))

print('IL-2 helices found:')
for i, (s, e) in enumerate(runs, 1):
    print(f'  H{i}: {s}-{e}  ({e-s+1} res)')

# --- per-residue map: helix / contact type ---
motif_set = set(df.loc[df.motif, 'resnum'])
alpha_set = set(df.loc[df.alpha_contact, 'resnum'])
print('\nmap (H=helix, *=motif contact, a=alpha contact):')
line1 = ''.join(ss.get(n, ' ') for n in nums)
line2 = ''.join('*' if n in motif_set else 'a' if n in alpha_set else '.'
                for n in nums)
for i in range(0, len(nums), 60):
    print(f'  {nums[i]:>4} {line1[i:i+60]}')
    print(f'       {line2[i:i+60]}')

# --- segments defined by helix membership ---
segs = []
for hs, he in runs:
    on_helix = sorted(r for r in motif_set if hs <= r <= he)
    if len(on_helix) >= 2:
        segs.append((max(hs, min(on_helix) - PAD),
                     min(he, max(on_helix) + PAD), len(on_helix)))

orphans = sorted(r for r in motif_set
                 if not any(s <= r <= e for s, e, _ in segs))

print('\nFrozen segments (helix-aware):')
tot = 0
for s, e, n in segs:
    sub = df[(df.resnum >= s) & (df.resnum <= e) & df.motif]
    print(f'  A{s}-{e}  ({e-s+1} res, {n} contacts, '
          f'ΔSASA {sub.d_beta.sum()+sub.d_gc.sum():.0f} Å²)')
    tot += e - s + 1
print(f'\nTotal frozen: {tot} residues in {len(segs)} segments')
print(f'Dropped orphan contacts: {orphans}')
print('\nCONTIG CORE:', '/'.join(f'A{s}-{e}' for s, e, _ in segs))

IL-2 helices found:
  H1: 7-27  (21 res)
  H2: 35-41  (7 res)
  H3: 53-61  (9 res)
  H4: 63-71  (9 res)
  H5: 82-97  (16 res)
  H6: 114-132  (19 res)

map (H=helix, *=motif contact, a=alpha contact):
     6 -HHHHHHHHHHHHHHHHHHHHH-------HHHHHHH------H----HHHHHHHHH-HHH
       ......**.**.***.**..........aa..a..aaaaa...............aa.aa
    66 HHHHHH------HHHHHHHHHHHHHHHH-------H----HHHHHHHHHHHHHHHHHHH-
       ..aa.aa.a..*..*..**..**..*.......a.a*........*...*..**.**..*

Frozen segments (helix-aware):
  A12-23  (12 res, 9 contacts, ΔSASA 428 Å²)
  A84-95  (12 res, 6 contacts, ΔSASA 303 Å²)
  A119-130  (12 res, 6 contacts, ΔSASA 295 Å²)

Total frozen: 36 residues in 3 segments
Dropped orphan contacts: [81, 110, 133]

CONTIG CORE: A12-23/A84-95/A119-130


In [ ]:
# Convert the frozen segments into RFdiffusion contig strings,
# with inter-segment Cα distances setting the minimum linker length,
# so that sampled topologies remain physically connectable
import json, numpy as np

TARGET_LENGTHS = [100, 120]     # total design lengths to sample
SEGS = [(12, 23), (84, 95), (119, 130)]

ca = {r.id[1]: r['CA'].get_coord() for r in model[A]
      if is_aa(r, standard=True) and 'CA' in r}

# --- how far apart are the segment termini in space? ---
print('inter-segment geometry (determines minimum linker length):')
min_link = []
for (s1, e1), (s2, e2) in zip(SEGS, SEGS[1:]):
    d = np.linalg.norm(ca[e1] - ca[s2])
    n = int(np.ceil(d / 3.0))          # ~3 Å per residue, generous
    min_link.append(n)
    print(f'  A{e1} → A{s2}:  {d:.1f} Å  → at least ~{n} residues')

# --- build contig strings ---
def contig(segs, total, pad=0):
    segs = [(max(1, s - pad), e + pad) for s, e in segs]
    frozen = sum(e - s + 1 for s, e in segs)
    free = total - frozen
    n_term, c_term = 6, 4
    body = free - n_term - c_term
    per = max(min_link) and body // (len(segs) - 1)
    parts = [f'{n_term-3}-{n_term+5}']
    for i, (s, e) in enumerate(segs):
        parts.append(f'A{s}-{e}')
        if i < len(segs) - 1:
            lo = max(min_link[i], per - 6)
            parts.append(f'{lo}-{per+6}')
    parts.append(f'{c_term-2}-{c_term+8}')
    return '/'.join(parts), frozen

print('\ncontigs to sample:')
contigs = {}
for pad in (0, 2):
    for total in TARGET_LENGTHS:
        c, f = contig(SEGS, total, pad)
        key = f'pad{pad}_len{total}'
        contigs[key] = c
        print(f'  {key:>14}  frozen {f:>2}  [{c}]')

# --- motif reference PDB for RFdiffusion ---
motif_pdb = f'{PROJ}/stage1_motif/il2_motif_chainA.pdb'
io.save(motif_pdb, Sel([A]))
print(f'\nmotif reference PDB: {motif_pdb}')

# --- params + summary ---
# Orphan list and segment boundaries are written as literals here, transcribed from
# cell 6 rather than passed programmatically. The values match that cell's computed
# output for this run; the dependency is not enforced in code
params = {
    'pdb_id': PDB_ID, 'chains': CHAIN,
    'contact_cutoff_A2': CONTACT_CUTOFF, 'alpha_cutoff_A2': ALPHA_CUTOFF,
    'min_helix': MIN_HELIX,
    'segments': SEGS, 'dropped_orphans': [81, 110, 133],
    'n_beta_contacts': int(n_b), 'n_gc_contacts': int(n_c),
    'n_alpha_contacts': int(n_a), 'beta_gc_alpha_overlap': int(overlap),
    'alpha_footprint_A2': float(alpha_res.d_alpha.sum()),
    'contigs': contigs,
}
with open(f'{PROJ}/stage1_motif/params.json', 'w') as fh:
    json.dump(params, fh, indent=2)
print(json.dumps(params, indent=2)[:400], '...')

inter-segment geometry (determines minimum linker length):
  A23 → A84:  12.8 Å  → at least ~5 residues
  A95 → A119:  21.1 Å  → at least ~8 residues

contigs to sample:
     pad0_len100  frozen 36  [3-11/A12-23/21-33/A84-95/21-33/A119-130/2-12]
     pad0_len120  frozen 36  [3-11/A12-23/31-43/A84-95/31-43/A119-130/2-12]
     pad2_len100  frozen 48  [3-11/A10-25/15-27/A82-97/15-27/A117-132/2-12]
     pad2_len120  frozen 48  [3-11/A10-25/25-37/A82-97/25-37/A117-132/2-12]

motif reference PDB: /content/drive/MyDrive/il2_mimetic/stage1_motif/il2_motif_chainA.pdb
{
  "pdb_id": "2B5I",
  "chains": {
    "IL2": "A",
    "IL2Rb": "B",
    "gc": "C",
    "IL2Ra": "D"
  },
  "contact_cutoff_A2": 10.0,

... [1 lines trimmed] ...
      12,
      23
    ],
    [
      84,
      95
    ],
    [
      119,
      130
    ]
  ],
  "dropped_orphans": [
    81,
    110,
    133
  ],
  "n_beta_contacts": 14,
  "n_gc_contacts": 12,
  ...


### Results

1. No IL-2 residue was detected contributing to both interfaces. This is conditional on the incomplete α structure.
2. Contact with an IL-2Rα has demonstrated to span over 849 Å² across 19 residues at positions 34-74 and 107-109.
3. Geometry: 24 residues satisfy the motif criterion. Three of them, 81, 110 and 133, are isolated contacts not lying on a helix with any neighbour, and were dropped: freezing a single residue in isolation specifies a geometric constraint that a designed scaffold has no reliable way to satisfy, and the cost of dropping them is 3 of 24 contacts. The remaining 21 contacts fall on three helices, yielding three segments of 12 residues each:

    | segment | contacts | ΔSASA buried |
    |---|---|---|
    | A12-23 | 9 | 428 Å² |
    | A84-95 | 6 | 303 Å² |
    | A119-130 | 6 | 295 Å² |

    These correspond to IL-2 helices A, C and D. The same three helices carry the receptor interface in Neo-2/15, which was generated independently from the structure.

###

#### Limitations:
1. The α analysis uses an incomplete receptor. 122 of 217 α residues are observed. The proximity test above bounds the risk but does not eliminate it; a definitive answer would require a complete α structure. This is revisited in later stage (5), where IL-2Rα is modelled full-length from sequence.
2. ΔSASA is geometry, not energy.


#### Comment to the notebook:
1. Remove hard-coded variables (e. g. n_term, c_term = 6, 4)